### Inserts, Batches & Knowledge Time

This notebook covers TimeDB's insert model in depth:
1. **Scalar `knowledge_time`** — one timestamp broadcast to all rows (quickstart pattern)
2. **Per-row `knowledge_time`** — VERSIONED insert, multiple forecast runs in one call
3. **Batch model** — one batch per `insert()` call; inspect history with `list_batches()`
4. **Read patterns** — `read()` for latest values, `read(overlapping=True)` for full forecast history
5. **VERSIONED inserts into flat series** — per-row knowledge_time for auditable fact data

In [1]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass  # Not running in Google Colab


In [2]:
import timedb as td
from timedb import TimeSeriesPolars
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

td.delete()
td.create()

Creating database schema...
✓ Schema created successfully


In [3]:
td.create_series("wind_forecast", unit="MW", overlapping=True,
                 description="Hourly wind power forecast")
td.create_series("meter_reading", unit="kWh",
                 description="Energy meter — flat series for auditable fact data")

for s in td.get_series().list_series():
    print(f"  {s['name']}: unit={s['unit']}  overlapping={s['overlapping']}")

  meter_reading: unit=kWh  overlapping=False
  wind_forecast: unit=MW  overlapping=True


## Part 1: Scalar `knowledge_time`

Pass `knowledge_time=` as a keyword argument to use one single `knowledge_time` timestamp to **all rows**.


In [4]:
np.random.seed(0)
base = datetime(2025, 1, 1, tzinfo=timezone.utc)

# Forecast run 1: issued at 00:00
kt1 = base
ts1 = TimeSeriesPolars.from_pandas(
    pd.DataFrame({
        "valid_time": pd.date_range("2025-01-01", periods=24, freq="h", tz="UTC"),
        "value": np.round(80 + 20 * np.sin(2 * np.pi * np.arange(24) / 24), 1),
    }),
    unit="MW",
)
result1 = td.get_series("wind_forecast").insert(ts1, knowledge_time=kt1)

print(f"InsertResult:")
print(f"  batch_id:  {result1.batch_id}")
print(f"  series_id: {result1.series_id}")
print(f"  workflow_id: {result1.workflow_id}")

InsertResult:
  batch_id:  019cfaa7-532e-711c-b1fc-179fb09cf035
  series_id: 1
  workflow_id: sdk-workflow


## Part 2: Per-row `knowledge_time` — VERSIONED Insert

Include a `knowledge_time` column in the DataFrame to assign a **different timestamp to each row**.
The SDK detects this column and the `TimeSeriesPolars` shape becomes `VERSIONED`.


In [5]:
np.random.seed(1)

# Build 3 forecast runs (issued every 12 h) in a single DataFrame
rows = []
for i, hours_offset in enumerate([12, 24, 36]):
    kt = base + timedelta(hours=hours_offset)
    np.random.seed(10 + i)
    values = np.round(
        80 + 20 * np.sin(2 * np.pi * np.arange(24) / 24) + np.random.normal(0, 3, 24), 1
    )
    for j, v in enumerate(values):
        rows.append({
            "knowledge_time": kt,
            "valid_time": datetime(2025, 1, 2, j, tzinfo=timezone.utc),
            "value": float(v),
        })

ts_versioned = TimeSeriesPolars.from_pandas(pd.DataFrame(rows), unit="MW")
result2 = td.get_series("wind_forecast").insert(ts_versioned)

print(f"Inserted {len(ts_versioned)} rows across 3 forecast runs in one insert() call")
print(f"DataShape:       {ts_versioned.shape}   ← VERSIONED because of knowledge_time column")
print(f"One batch_id:    {result2.batch_id}")
print(f"(result1 batch:  {result1.batch_id})")

Inserted 72 rows across 3 forecast runs in one insert() call
DataShape:       DataShape.VERSIONED   ← VERSIONED because of knowledge_time column
One batch_id:    019cfaa7-534f-762f-ab32-caaab43ae771
(result1 batch:  019cfaa7-532e-711c-b1fc-179fb09cf035)


## Part 3: `list_batches()` — Inspect Batch History

`list_batches()` discovers all batches that contributed data to a series. Results are ordered by `inserted_at` DESC — most recent first.


In [6]:
batches = td.get_series("wind_forecast").list_batches()
print(f"Found {len(batches)} batches for wind_forecast:\n")
for b in batches:
    print(f"  batch_id   = {b['batch_id']}")
    print(f"  inserted_at= {b['inserted_at']}")
    print(f"  workflow_id= {b['workflow_id']}")
    print()

Found 2 batches for wind_forecast:

  batch_id   = 019cfaa7-534f-762f-ab32-caaab43ae771
  inserted_at= 2026-03-17 07:16:35.791227+00:00
  workflow_id= sdk-workflow

  batch_id   = 019cfaa7-532e-711c-b1fc-179fb09cf035
  inserted_at= 2026-03-17 07:16:35.759053+00:00
  workflow_id= sdk-workflow



In [7]:
# Cross-reference with InsertResult: batch_ids must match
batch_ids = [b["batch_id"] for b in batches]

assert batches[0]["batch_id"] == result2.batch_id, "most recent batch should be result2"
assert result1.batch_id in batch_ids, "result1 batch should appear in history"

print("✓ result2.batch_id == batches[0]['batch_id']  (most recent first)")
print("✓ result1.batch_id found in batch history")

✓ result2.batch_id == batches[0]['batch_id']  (most recent first)
✓ result1.batch_id found in batch history


## Part 4: Reading Back

Two read modes:

| Call | Returns | Shape | Use case |
|------|---------|-------|----------|
| `read()` | latest value per `valid_time` | `SIMPLE` | current best estimate |
| `read(overlapping=True)` | latest correction per `(knowledge_time, valid_time)` | `VERSIONED` | full forecast history |

In [8]:
# read() — latest value per valid_time (highest knowledge_time wins)
ts_latest = td.get_series("wind_forecast").read(
    start_valid=datetime(2025, 1, 2, tzinfo=timezone.utc),
    end_valid=datetime(2025, 1, 3, tzinfo=timezone.utc),
)
display(ts_latest)
print(f"→ {ts_latest.num_rows} rows, one per valid_time")

Name,wind_forecast
Shape,SIMPLE
Rows,24
Timezone,UTC
Unit,MW
Description,Hourly wind power forecast
Timeseries type,OVERLAPPING
valid_time,wind_forecast
2025-01-02 00:00,81.4
2025-01-02 01:00,83.1
2025-01-02 02:00,90.7


→ 24 rows, one per valid_time


In [9]:
# read(overlapping=True) — one row per (knowledge_time, valid_time)
ts_all = td.get_series("wind_forecast").read(
    start_valid=datetime(2025, 1, 2, tzinfo=timezone.utc),
    end_valid=datetime(2025, 1, 3, tzinfo=timezone.utc),
    overlapping=True,
)
display(ts_all)

df_all = ts_all.to_pandas()
n_kts = df_all.index.get_level_values("knowledge_time").nunique()
print(f"→ {ts_all.num_rows} rows across {n_kts} forecast runs")

TimeSeriesPolars
┌─────────────────────────────────────────────────────┐
│  Name:             wind_forecast                    │
│  Shape:            VERSIONED                        │
│  Rows:             72                               │
│  Timezone:         UTC                              │
│  Unit:             MW                               │
│  Description:      Hourly wind power forecast       │
│  Timeseries type:  OVERLAPPING                      │
├─────────────────────────────────────────────────────┤
│                                      wind_forecast  │
│  2025-01-01 12:00, 2025-01-02 00:00           84.0  │
│  2025-01-01 12:00, 2025-01-02 01:00           87.3  │
│  2025-01-01 12:00, 2025-01-02 02:00           85.4  │
│  ...                                           ...  │
│  2025-01-02 12:00, 2025-01-02 21:00           66.3  │
│  2025-01-02 12:00, 2025-01-02 22:00           71.9  │
│  2025-01-02 12:00, 2025-01-02 23:00           76.4  │
└─────────────────────────────────────────────────────┘

→ 72 rows across 3 forecast runs


## Part 5: VERSIONED Insert into a Flat Series

Flat series (overlapping=False) also accept per-row `knowledge_time` columns.
This is useful when inserting auditable fact data where you want to record
*when* each measurement was taken — not just its value.

The flat table stores `knowledge_time` per-row and uses an upsert on `(series_id, valid_time)`.
The row with the **highest `knowledge_time`** wins for a given `valid_time`.

In [10]:
# Each measurement is a separate insert() call — the flat table upserts on (series_id, valid_time)
# so the second insert supersedes the first for the same valid_time.

# Reading 1 — recorded at 08:00
td.get_series("meter_reading").insert(
    TimeSeriesPolars.from_pandas(
        pd.DataFrame([{"valid_time": datetime(2025, 1, 1, 6, tzinfo=timezone.utc), "value": 142.3}]),
        unit="kWh",
    ),
    knowledge_time=datetime(2025, 1, 1, 8, tzinfo=timezone.utc),
)
print("Inserted reading 1: 142.3 kWh at 06:00  (recorded 08:00)")

# Reading 2 — recorded at 10:00, supersedes first via upsert
td.get_series("meter_reading").insert(
    TimeSeriesPolars.from_pandas(
        pd.DataFrame([{"valid_time": datetime(2025, 1, 1, 6, tzinfo=timezone.utc), "value": 142.8}]),
        unit="kWh",
    ),
    knowledge_time=datetime(2025, 1, 1, 10, tzinfo=timezone.utc),
)
print("Inserted reading 2: 142.8 kWh at 06:00  (recorded 10:00)")

# read() returns the current (latest-inserted) value
df_flat = td.get_series("meter_reading").read().to_pandas()
print(f"\nCurrent reading at 06:00: {df_flat['value'].iloc[0]} kWh")
print("  (142.8 — the 10:00 insert superseded the 08:00 insert via upsert)")

Inserted reading 1: 142.3 kWh at 06:00  (recorded 08:00)
Inserted reading 2: 142.8 kWh at 06:00  (recorded 10:00)

Current reading at 06:00: 142.8 kWh
  (142.8 — the 10:00 insert superseded the 08:00 insert via upsert)


### Summary

| Pattern | How | DataShape | Batches created |
|---------|-----|-----------|------------------|
| Scalar `knowledge_time` | `insert(ts, knowledge_time=dt)` | `SIMPLE` | 1 per call |
| Per-row `knowledge_time` | column in DataFrame / `TimeSeriesPolars` | `VERSIONED` | 1 per call |

- **One batch per `insert()` call** — `InsertResult.batch_id` is always a single UUIDv7
- `list_batches()` returns batch history for a series, most recent first
- `read()` → latest value per `valid_time` (`SIMPLE` shape)
- `read(overlapping=True)` → all forecast runs (`VERSIONED` shape, `(knowledge_time, valid_time)` index)
- Per-row `knowledge_time` works on **both flat and overlapping** series